# 🧪 BabyChatty RAG System — Evaluation Notebook v2

**UC3M · NLP Final Project 2025/2026**

This notebook validates all major design decisions of the BabyChatty production pipeline
and generates evidence for the final report. The pipeline tested here matches **`chat.py`**
exactly: two-stage retrieval (similarity-score threshold → cross-encoder reranker), CoT
JSON output, and automatic query translation.

### Experiments

| # | Parameter | Values tested | RAGAS? | Sample | Est. time |
|---|-----------|--------------|--------|--------|-----------|
| A | Similarity threshold (`ret_threshold`) | 0.30 / 0.40 / **0.45** / 0.55 | ❌ | 20q | ~25–40 min |
| B | Candidate pool (`max_ret_num`) | 5 / 10 / **20** / 30 | ❌ | 20q | ~25–40 min |
| C | Final top-N (`retrieval_num`) | 3 / **5** / 7 / 10 | ❌ | 20q | ~25–40 min |
| D | Temperature | 0.0 / **0.3** / 0.6 / 1.0 | ✅ | 20q | ~60–90 min |
| E | Reranker model | **bge-reranker-large** vs bge-reranker-base | ✅ | 20q | ~30–50 min |
| F | Embedding model | **bge-large-en-v1.5** vs bge-m3 | ✅ | 70q | ~90–150 min |
| G | Multilingual breakdown | per-language RAGAS | ✅ | uses F results | — |

> **⏱️ Total estimated runtime: ~3–5 hours** (optimized from the naive ~10–18h).
> Experiments A/B/C use `run_ragas=False` because Coverage and Hallucination Rate
> — the metrics that matter most for retrieval tuning — come from binary classification
> and don't require the judge LLM. RAGAS is reserved for D/E/F where answer quality
> is the main variable.

**Metrics (per project rubric Table 1):**
- 📚 **Document Coverage** — % of in-scope queries answered
- ⏱️ **Response Time** — seconds per query
- 🧠 **RAG System Quality** — Faithfulness, Answer Relevancy, Context Utilization (RAGAS)
- 🔴 **Hallucination Rate** — FP/(FP+TN) — model answered an out-of-scope question
- 📉 **Classification** — Accuracy, Precision, Recall, F1

---

## 0. Setup & Imports

In [ ]:
import os, sys, time, re, json, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from itertools import product as iterproduct

warnings.filterwarnings('ignore')
load_dotenv()

# ── Project root ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent   # adjust if notebook lives in /notebooks
sys.path.insert(0, str(PROJECT_ROOT))

# ── Plotting defaults ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS = {
    'main':   '#4db8b0',   # teal  — primary model
    'alt':    '#f4845f',   # coral — alternative model
    'accent': '#9b7fca',   # purple
    'ok':     '#4caf50',   # green
    'warn':   '#e86c47',   # orange-red
    'neutral':'#aaaaaa',   # grey
}

# ── Output dirs ───────────────────────────────────────────────────────────────
FIGURES_DIR = PROJECT_ROOT / 'figures'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print('✅ Imports OK')
print(f'   Project root : {PROJECT_ROOT}')
print(f'   Figures dir  : {FIGURES_DIR}')
print(f'   Results dir  : {RESULTS_DIR}')

In [ ]:
import ollama
import torch
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.documents import Document
from langchain_core.callbacks import Callbacks
from langchain_openai import ChatOpenAI
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, ContextUtilization
from datasets import Dataset
from typing import Sequence, Optional

from utils import GenConfig as cfg, VectorDBFactory

print('✅ Project modules loaded')
print(f'   LLM          : {cfg.model_name}')
print(f'   Judge        : {cfg.judge_name}')
print(f'   Embedding    : {cfg.embedding_model}')
print(f'   Reranker     : {cfg.re_rank_model}')
print(f'   Ollama URL   : {cfg.ollama_url}')

## 1. Load Evaluation Dataset

The dataset (`eval_questions_v2_transformed.csv`) contains **140 questions** across 7 languages:

| Label | Meaning | Count |
|-------|---------|-------|
| `1`   | In-scope pediatric question | 100 |
| `0`   | Out-of-scope / off-topic     | 40  |

In [ ]:
EVAL_CSV = PROJECT_ROOT / 'data' / 'eval_questions_v2_transformed.csv'
df_eval  = pd.read_csv(EVAL_CSV)

print(f'Total questions: {len(df_eval)}')
dist = df_eval.groupby(['language', 'label']).size().reset_index(name='count')
print(dist.to_string(index=False))
df_eval.head(8)

## 2. Shared Helpers

All evaluation logic is defined here so every experiment section can run independently.

In [ ]:
# ── Ollama client ─────────────────────────────────────────────────────────────
client = ollama.Client(
    host=cfg.ollama_url,
    headers={'X-API-KEY': os.getenv('OLLAMA_API_KEY')},
)

# ── RAGAS judge LLM ──────────────────────────────────────────────────────────
eval_llm = ChatOpenAI(
    base_url=f"{cfg.ollama_url}/v1",
    api_key='fake-key',
    model=cfg.judge_name,
    default_headers={'X-API-KEY': os.getenv('OLLAMA_API_KEY')},
    temperature=0,
    timeout=120,
)

# ── Language detection ────────────────────────────────────────────────────────
LANG_MAP = {'es':'Spanish','en':'English','fr':'French','de':'German',
            'it':'Italian','pt':'Portuguese','ca':'Catalan'}

def detect_language(text: str) -> str:
    try:
        return LANG_MAP.get(detect(text), 'English')
    except LangDetectException:
        return 'English'

# ── Negative-answer detection (mirrors cfg.no_info_patterns) ─────────────────
def is_negative(answer: str, docs: list) -> bool:
    if not docs:
        return True
    return any(re.search(p, answer.lower()) for p in cfg.no_info_patterns)

# ── JSON CoT parser (mirrors _parse_cot_response in chat.py) ─────────────────
def parse_cot(raw: str) -> dict:
    cleaned = raw.strip()
    if cleaned.startswith('```'):
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
        cleaned = re.sub(r'\s*```$', '', cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            a = parsed.get('answer', '')
            r = parsed.get('reasoning', '')
            if isinstance(a, list): a = '\n'.join(str(x) for x in a)
            if isinstance(r, dict): r = ' | '.join(f"{k}: {v}" for k,v in r.items())
            if a: return {'reasoning': str(r), 'answer': str(a)}
    except Exception:
        pass
    # Regex fallback
    a_m = re.search(r'"answer"\s*:\s*"((?:[^"\\]|\\.)*)"', cleaned, re.DOTALL)
    r_m = re.search(r'"reasoning"\s*:\s*"((?:[^"\\]|\\.)*)"', cleaned, re.DOTALL)
    return {
        'reasoning': r_m.group(1).replace('\\n','\n') if r_m else '',
        'answer':    a_m.group(1).replace('\\n','\n') if a_m else raw.strip(),
    }


# ── Custom reranker (mirrors CustomCrossEncoderReranker in chat.py) ───────────
class CustomCrossEncoderReranker(CrossEncoderReranker):
    """Persists relevance scores in document metadata."""
    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Optional[Callbacks] = None,
    ) -> Sequence[Document]:
        if not documents:
            return []
        scores = self.model.score([(query, d.page_content) for d in documents])
        ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
        return [
            Document(page_content=d.page_content,
                     metadata={**d.metadata, 'relevance_score': float(s)})
            for d, s in ranked[:self.top_n]
        ]


# ── Build retriever for a given param combo ───────────────────────────────────
def build_retriever(
    vectordb,
    reranker_model,
    score_threshold: float,
    max_ret_num: int,
    top_n: int,
):
    """Construct the two-stage retriever matching the production pipeline."""
    base = vectordb.as_retriever(
        search_type='similarity_score_threshold',
        search_kwargs={'score_threshold': score_threshold, 'k': max_ret_num},
    )
    compressor = CustomCrossEncoderReranker(model=reranker_model, top_n=top_n)
    return ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=base
    )


# ── Single RAG turn (mirrors _get_ai_response in chat.py) ────────────────────
def run_rag_turn(
    question: str,
    retriever,
    temperature: float,
) -> dict:
    t0 = time.time()
    lang = detect_language(question)

    # Translate non-English queries (matches production pipeline)
    query_for_retrieval = question
    if lang != 'English':
        try:
            query_for_retrieval = GoogleTranslator(source='auto', target='en').translate(question)
        except Exception:
            pass  # fallback: use original

    docs = retriever.invoke(query_for_retrieval)

    context_parts = []
    for i, doc in enumerate(docs, 1):
        title  = doc.metadata.get('title',  'Unknown')
        source = doc.metadata.get('source', 'Unknown')
        context_parts.append(f'[Source {i}: {title} | {source}]\n{doc.page_content}')
    context = '\n\n'.join(context_parts)

    final_prompt = cfg.prompt_template.format(
        lang=lang, context=context, question=question
    )

    response = client.chat(
        model=cfg.model_name,
        messages=[{'role': 'user', 'content': final_prompt}],
        options={'temperature': temperature},
    )
    raw    = response['message']['content']
    parsed = parse_cot(raw)
    answer = parsed['answer']

    return {
        'answer':   answer,
        'docs':     docs,
        'lang':     lang,
        'time':     time.time() - t0,
        'negative': is_negative(answer, docs),
    }


# ── RAGAS evaluation ──────────────────────────────────────────────────────────
def ragas_eval(question: str, answer: str, docs: list, emb_model) -> dict:
    contexts = [d.page_content for d in docs]
    ds = Dataset.from_dict({
        'question': [question],
        'answer':   [answer],
        'contexts': [contexts],
    })
    try:
        res = evaluate(
            dataset=ds,
            metrics=[faithfulness, answer_relevancy, ContextUtilization()],
            llm=eval_llm,
            embeddings=emb_model,
            show_progress=False,
        )
        df_r = res.to_pandas()
        return {
            'faithfulness':        float(df_r['faithfulness'].iloc[0]),
            'answer_relevancy':    float(df_r['answer_relevancy'].iloc[0]),
            'context_utilization': float(df_r['context_utilization'].iloc[0]),
        }
    except Exception as e:
        print(f'  [RAGAS error] {e}')
        return {'faithfulness': np.nan, 'answer_relevancy': np.nan,
                'context_utilization': np.nan}


# ── Full batch evaluation ─────────────────────────────────────────────────────
def run_experiment(
    questions_df: pd.DataFrame,
    retriever,
    emb_model,
    temperature: float,
    run_ragas: bool = True,
    sample_n: int = None,
    seed: int = 42,
) -> pd.DataFrame:
    if sample_n:
        questions_df = questions_df.sample(n=sample_n, random_state=seed).reset_index(drop=True)

    rows = []
    for _, row in tqdm(questions_df.iterrows(), total=len(questions_df)):
        q     = row['question']
        label = int(row['label'])

        result = run_rag_turn(q, retriever, temperature)
        pred   = 0 if result['negative'] else 1

        ragas_scores = {'faithfulness': np.nan, 'answer_relevancy': np.nan,
                        'context_utilization': np.nan}
        if run_ragas and not result['negative']:
            ragas_scores = ragas_eval(q, result['answer'], result['docs'], emb_model)

        rows.append({
            'id':                  row.get('id', ''),
            'question':            q,
            'language':            row.get('language', result['lang']),
            'category':            row.get('category', ''),
            'label':               label,
            'pred':                pred,
            'response_time':       result['time'],
            'n_docs_retrieved':    len(result['docs']),
            'hallucination_flag':  int(pred == 1 and label == 0),
            **ragas_scores,
            'answer_preview':      result['answer'][:200],
        })

    return pd.DataFrame(rows)


# ── Aggregate metrics ─────────────────────────────────────────────────────────
def summarize(df: pd.DataFrame, label: str = '') -> dict:
    total = len(df)
    TP = int(((df['label']==1) & (df['pred']==1)).sum())
    FP = int(((df['label']==0) & (df['pred']==1)).sum())
    TN = int(((df['label']==0) & (df['pred']==0)).sum())
    FN = int(((df['label']==1) & (df['pred']==0)).sum())

    acc   = (TP+TN)/total  if total      else 0
    prec  = TP/(TP+FP)     if (TP+FP)   else 0
    rec   = TP/(TP+FN)     if (TP+FN)   else 0
    f1    = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    cov   = TP/(TP+FN)     if (TP+FN)   else 0
    hall  = FP/(FP+TN)     if (FP+TN)   else 0

    def _m(col):
        v = df[col].dropna()
        return float(v.mean()) if len(v) else np.nan

    faith = _m('faithfulness')
    rel   = _m('answer_relevancy')
    ctx   = _m('context_utilization')
    qual  = float(np.nanmean([faith, rel, ctx]))

    return {
        'label': label,
        'n_total': total, 'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
        'accuracy':           round(acc,  3),
        'precision':          round(prec, 3),
        'recall':             round(rec,  3),
        'f1':                 round(f1,   3),
        'document_coverage':  round(cov,  3),
        'hallucination_rate': round(hall, 3),
        'avg_response_time':  round(df['response_time'].mean(), 2),
        'avg_docs_retrieved': round(df['n_docs_retrieved'].mean(), 2),
        'faithfulness':       round(faith,3) if not np.isnan(faith) else np.nan,
        'answer_relevancy':   round(rel,  3) if not np.isnan(rel)   else np.nan,
        'context_utilization':round(ctx,  3) if not np.isnan(ctx)   else np.nan,
        'rag_system_quality': round(qual, 3) if not np.isnan(qual)  else np.nan,
    }


# ── Plotting helper ───────────────────────────────────────────────────────────
METRICS_DISPLAY = [
    ('rag_system_quality',  '🧠 RAG Quality',     COLORS['main']),
    ('document_coverage',   '📚 Doc Coverage',    COLORS['ok']),
    ('hallucination_rate',  '🔴 Hallucination',   COLORS['warn']),
    ('avg_response_time',   '⏱️ Response Time (s)', COLORS['accent']),
]

def bar_comparison(summaries: list[dict], x_labels: list[str],
                   title: str, fname: str):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, (metric, m_title, color) in zip(axes, METRICS_DISPLAY):
        vals = [s[metric] for s in summaries]
        bars = ax.bar(x_labels, vals, color=color, alpha=0.85, edgecolor='white')
        ax.set_title(m_title, fontsize=10, fontweight='bold')
        ax.set_ylim(0, max([v for v in vals if not np.isnan(v)] or [1]) * 1.3)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    path = FIGURES_DIR / fname
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'📊 Saved → {path}')

print('✅ All helpers defined')

## 3. Load Production Vector DB & Reranker Model

In [ ]:
# ── Embedding model (production: BAAI/bge-large-en-v1.5) ─────────────────────
print(f'Loading embedding model: {cfg.embedding_model} ...')
emb_prod = HuggingFaceEmbeddings(
    model_name=cfg.embedding_model,
    model_kwargs={'device': cfg.emb_device},
    encode_kwargs={'batch_size': 1},
)

# ── Chroma vector DB ──────────────────────────────────────────────────────────
print(f'Loading Chroma DB from {cfg.chroma_dir} ...')
vectordb = Chroma(
    persist_directory=str(cfg.chroma_dir),
    embedding_function=emb_prod,
)
n_chunks = vectordb._collection.count()
print(f'  ✅ DB ready — {n_chunks:,} chunks')

# ── Cross-encoder reranker models ─────────────────────────────────────────────
print(f'Loading reranker: {cfg.re_rank_model} ...')
reranker_large = HuggingFaceCrossEncoder(
    model_name=cfg.re_rank_model,       # BAAI/bge-reranker-large (production)
    model_kwargs={'device': cfg.emb_device},
)

RERANKER_BASE_NAME = 'BAAI/bge-reranker-base'
print(f'Loading reranker: {RERANKER_BASE_NAME} ...')
reranker_base = HuggingFaceCrossEncoder(
    model_name=RERANKER_BASE_NAME,
    model_kwargs={'device': cfg.emb_device},
)

print('✅ All models loaded')

---
## Experiment A — Similarity Score Threshold (`ret_threshold`)

**Hypothesis:** A low threshold admits noisy, weakly-related chunks → higher hallucination.
A very high threshold starves the reranker and increases false negatives.
We test `{0.30, 0.40, 0.45, 0.55}` with `max_ret_num=20`, `top_n=5`, `temperature=0.3`.

**RAGAS disabled** — Coverage and Hallucination Rate from binary classification are sufficient
to tune the retrieval threshold. This keeps each run under ~6 min.

> ⏱️ Expected time: ~25–40 min (20-question sample × 4 thresholds, no RAGAS)

In [ ]:
# ── Balanced 20-question sample (15 positives + 5 negatives) ─────────────────
sample_pos_20 = df_eval[df_eval['label']==1].sample(n=15, random_state=42)
sample_neg_20 = df_eval[df_eval['label']==0].sample(n=5,  random_state=42)
df_sample_20  = pd.concat([sample_pos_20, sample_neg_20]).reset_index(drop=True)

THRESHOLDS   = [0.30, 0.40, 0.45, 0.55]
FIXED_MAX    = 20
FIXED_TOPN   = 5
FIXED_TEMP   = 0.3

results_thr = {}
for thr in THRESHOLDS:
    print(f'\n── threshold={thr} ───────────────────────────────────────────')
    ret = build_retriever(vectordb, reranker_large, thr, FIXED_MAX, FIXED_TOPN)
    df_res = run_experiment(df_sample_20, ret, emb_prod,
                            temperature=FIXED_TEMP, run_ragas=False)
    results_thr[thr] = df_res
    s = summarize(df_res, f'thr={thr}')
    print(f'  Coverage={s["document_coverage"]:.1%} | '
          f'Halluc={s["hallucination_rate"]:.1%} | '
          f'Quality={s["rag_system_quality"]:.3f} | '
          f'Docs/q={s["avg_docs_retrieved"]:.1f} | '
          f'Time={s["avg_response_time"]:.1f}s')

print('\n✅ Threshold experiment done')

In [ ]:
sums_thr = [summarize(results_thr[t], f'thr={t}') for t in THRESHOLDS]
df_thr   = pd.DataFrame(sums_thr)

SHOW_COLS = ['label','document_coverage','hallucination_rate','faithfulness',
             'answer_relevancy','context_utilization','rag_system_quality',
             'avg_docs_retrieved','avg_response_time']
print(df_thr[SHOW_COLS].to_string(index=False))

bar_comparison(sums_thr, [f'thr={t}' for t in THRESHOLDS],
               'Experiment A — Effect of Similarity Threshold',
               'expA_threshold.png')

### A — Interpretation

*Fill in after running — key points for the report:*
- Which threshold achieves the best balance between Coverage and Hallucination Rate?
- Does `avg_docs_retrieved` drop to 0 at threshold=0.55? If so, that threshold is too aggressive.
- The chosen value `cfg.ret_threshold=0.45` should sit at the Pareto frontier.
- Note the trade-off: lower threshold → more candidates for the reranker, but also more noise.

---
## Experiment B — Candidate Pool Size (`max_ret_num`)

**Hypothesis:** The reranker needs enough candidates to find the best 5. Too few starves it;
too many wastes compute without gain. We test `{5, 10, 20, 30}` with the best threshold from A.

> ⏱️ Expected time: ~12–18 min

In [ ]:
BEST_THRESHOLD = 0.45     # ← update from Experiment A results if needed
POOL_SIZES     = [5, 10, 20, 30]

results_pool = {}
for pool in POOL_SIZES:
    print(f'\n── max_ret_num={pool} ─────────────────────────────────────────')
    ret = build_retriever(vectordb, reranker_large, BEST_THRESHOLD, pool, FIXED_TOPN)
    df_res = run_experiment(df_sample_20, ret, emb_prod,
                            temperature=FIXED_TEMP, run_ragas=False)
    results_pool[pool] = df_res
    s = summarize(df_res, f'pool={pool}')
    print(f'  Coverage={s["document_coverage"]:.1%} | '
          f'Halluc={s["hallucination_rate"]:.1%} | '
          f'Quality={s["rag_system_quality"]:.3f} | '
          f'Docs/q={s["avg_docs_retrieved"]:.1f} | '
          f'Time={s["avg_response_time"]:.1f}s')

print('\n✅ Pool-size experiment done')

In [ ]:
sums_pool = [summarize(results_pool[p], f'pool={p}') for p in POOL_SIZES]
df_pool   = pd.DataFrame(sums_pool)
print(df_pool[SHOW_COLS].to_string(index=False))

bar_comparison(sums_pool, [f'pool={p}' for p in POOL_SIZES],
               'Experiment B — Candidate Pool Size Before Reranking',
               'expB_pool_size.png')

### B — Interpretation

*Fill in after running:*
- Does quality plateau between pool=20 and pool=30? That justifies `cfg.max_ret_num=20`.
- Does response time grow significantly with pool size? (It shouldn't if reranking dominates.)
- Very small pools (5) risk the reranker never seeing a relevant chunk.

---
## Experiment C — Final Top-N After Reranking (`retrieval_num`)

**Hypothesis:** The LLM context window is not a bottleneck at chunk_size=1000, but passing
too many chunks introduces noise. We test `top_n ∈ {3, 5, 7, 10}` with the best A+B params.

> ⏱️ Expected time: ~12–18 min

In [ ]:
BEST_POOL  = 20    # ← update from Experiment B
TOPN_VALUES = [3, 5, 7, 10]

results_topn = {}
for topn in TOPN_VALUES:
    print(f'\n── top_n={topn} ──────────────────────────────────────────────')
    ret = build_retriever(vectordb, reranker_large, BEST_THRESHOLD, BEST_POOL, topn)
    df_res = run_experiment(df_sample_20, ret, emb_prod,
                            temperature=FIXED_TEMP, run_ragas=False)
    results_topn[topn] = df_res
    s = summarize(df_res, f'top_n={topn}')
    print(f'  Coverage={s["document_coverage"]:.1%} | '
          f'Halluc={s["hallucination_rate"]:.1%} | '
          f'Quality={s["rag_system_quality"]:.3f} | '
          f'Time={s["avg_response_time"]:.1f}s')

print('\n✅ Top-N experiment done')

In [ ]:
sums_topn = [summarize(results_topn[n], f'top_n={n}') for n in TOPN_VALUES]
df_topn   = pd.DataFrame(sums_topn)
print(df_topn[SHOW_COLS].to_string(index=False))

bar_comparison(sums_topn, [f'top_n={n}' for n in TOPN_VALUES],
               'Experiment C — Final Top-N Chunks Passed to LLM',
               'expC_topn.png')

---
## Experiment D — Temperature

**Hypothesis:** Higher temperature increases creativity but risks hallucination in a factual
medical context. Production uses `temperature=0.3` as a middle ground.
We test `{0.0, 0.3, 0.6, 1.0}` with the best retrieval config from A–C.

> ⏱️ Expected time: ~12–18 min

In [ ]:
BEST_TOPN  = 5    # ← update from Experiment C
TEMPS      = [0.0, 0.3, 0.6, 1.0]

ret_best = build_retriever(vectordb, reranker_large, BEST_THRESHOLD, BEST_POOL, BEST_TOPN)

results_temp = {}
for temp in TEMPS:
    print(f'\n── temperature={temp} ────────────────────────────────────────')
    df_res = run_experiment(df_sample_20, ret_best, emb_prod,
                            temperature=temp, run_ragas=True)
    results_temp[temp] = df_res
    s = summarize(df_res, f'temp={temp}')
    print(f'  Coverage={s["document_coverage"]:.1%} | '
          f'Halluc={s["hallucination_rate"]:.1%} | '
          f'Faithfulness={s["faithfulness"]:.3f} | '
          f'Quality={s["rag_system_quality"]:.3f}')

print('\n✅ Temperature experiment done')

In [ ]:
sums_temp = [summarize(results_temp[t], f'temp={t}') for t in TEMPS]
df_temp   = pd.DataFrame(sums_temp)
print(df_temp[SHOW_COLS].to_string(index=False))

bar_comparison(sums_temp, [f'temp={t}' for t in TEMPS],
               'Experiment D — Effect of Temperature on RAG Quality',
               'expD_temperature.png')

# ── Faithfulness vs Hallucination scatter ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
for s, t in zip(sums_temp, TEMPS):
    ax.scatter(s['faithfulness'], s['hallucination_rate'],
               s=180, zorder=5, label=f'T={t}',
               color=plt.cm.plasma(t))
    ax.annotate(f'T={t}', (s['faithfulness'], s['hallucination_rate']),
                textcoords='offset points', xytext=(6,4), fontsize=9)
ax.set_xlabel('Faithfulness ↑')
ax.set_ylabel('Hallucination Rate ↓')
ax.set_title('Temperature Trade-off: Faithfulness vs Hallucination', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'expD_tradeoff_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### D — Interpretation

*Fill in after running:*
- At `T=0.0` the model is fully deterministic. Does faithfulness peak there?
- `T=0.3` should keep high faithfulness while still allowing fluent, coherent phrasing.
- `T≥0.6` may increase hallucination; reference the scatter plot.
- This justifies `cfg.temperature=0.3` for a medical assistant that must be accurate
  but readable.

---
## Experiment E — Reranker Model Comparison

**Hypothesis:** `bge-reranker-large` provides better ranking quality than `bge-reranker-base`
at the cost of more compute. We run both on the 30-question sample with the best config.

> ⏱️ Expected time: ~8–12 min

In [ ]:
BEST_TEMP = 0.3    # ← update from Experiment D

ret_large = build_retriever(vectordb, reranker_large, BEST_THRESHOLD, BEST_POOL, BEST_TOPN)
ret_base  = build_retriever(vectordb, reranker_base,  BEST_THRESHOLD, BEST_POOL, BEST_TOPN)

print('── bge-reranker-large ───────────────────────────────────────────')
df_rer_large = run_experiment(df_sample_20, ret_large, emb_prod,
                              temperature=BEST_TEMP, run_ragas=True)
df_rer_large.to_csv(RESULTS_DIR / 'expE_reranker_large.csv', index=False)

print('\n── bge-reranker-base ────────────────────────────────────────────')
df_rer_base = run_experiment(df_sample_20, ret_base, emb_prod,
                             temperature=BEST_TEMP, run_ragas=True)
df_rer_base.to_csv(RESULTS_DIR / 'expE_reranker_base.csv', index=False)

print('\n✅ Reranker comparison done')

In [ ]:
s_rer_large = summarize(df_rer_large, 'bge-reranker-large')
s_rer_base  = summarize(df_rer_base,  'bge-reranker-base')

df_rer_cmp  = pd.DataFrame([s_rer_large, s_rer_base])
print(df_rer_cmp[SHOW_COLS].to_string(index=False))

# ── Side-by-side bar chart ────────────────────────────────────────────────────
compare_metrics = ['document_coverage','hallucination_rate','faithfulness',
                   'answer_relevancy','context_utilization','rag_system_quality']
labels_plot = ['Coverage','Halluc.\nRate','Faithfulness',
               'Answer\nRelevancy','Context\nUtil.','RAG\nQuality']

large_vals = [s_rer_large[m] for m in compare_metrics]
base_vals  = [s_rer_base[m]  for m in compare_metrics]
x = np.arange(len(compare_metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - w/2, large_vals, w, label='bge-reranker-large',
            color=COLORS['main'], alpha=0.88, edgecolor='white')
b2 = ax.bar(x + w/2, base_vals,  w, label='bge-reranker-base',
            color=COLORS['alt'],  alpha=0.88, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(labels_plot, fontsize=10)
ax.set_ylim(0, 1.2)
ax.set_title('Experiment E — Reranker Model Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylabel('Score')
for bar in list(b1)+list(b2):
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x()+bar.get_width()/2, h+0.02, f'{h:.2f}',
                ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'expE_reranker_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Response-time comparison
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['bge-reranker-large', 'bge-reranker-base'],
       [s_rer_large['avg_response_time'], s_rer_base['avg_response_time']],
       color=[COLORS['main'], COLORS['alt']], alpha=0.85, edgecolor='white')
ax.set_ylabel('Avg Response Time (s)')
ax.set_title('Reranker Model — Response Time', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'expE_reranker_time.png', dpi=150, bbox_inches='tight')
plt.show()

### E — Interpretation

*Fill in after running:*
- Does `bge-reranker-large` outperform `base` on faithfulness and coverage?
- How much extra latency does `large` add?
- If quality gain is small but latency doubles, `base` may be preferable in production.
- Justify the choice of `cfg.re_rank_model = 'BAAI/bge-reranker-large'`.

---
## Experiment F — Embedding Model Comparison

**Hypothesis:** `BAAI/bge-large-en-v1.5` (English-optimised, larger) should outperform
`BAAI/bge-m3` (multilingual, slightly smaller) on our English-dominant corpus.
However, bge-m3 may win on Spanish/French queries.

> ⚠️ This experiment requires a **separate Chroma DB** built with bge-m3 embeddings.
> If `chroma_db_bge_m3/` does not exist, follow the instructions in the cell below.
> Expected time: ~15–25 min (full 140 questions × 2 models)

In [ ]:
# ── Alternative embedding model ───────────────────────────────────────────────
BGE_M3_NAME    = 'BAAI/bge-m3'
CHROMA_BGE_M3  = PROJECT_ROOT / 'chroma_db_bge_m3'

print(f'Loading {BGE_M3_NAME} ...')
emb_m3 = HuggingFaceEmbeddings(
    model_name=BGE_M3_NAME,
    model_kwargs={'device': cfg.emb_device},
    encode_kwargs={'batch_size': 1},
)

if CHROMA_BGE_M3.exists():
    vectordb_m3 = Chroma(
        persist_directory=str(CHROMA_BGE_M3),
        embedding_function=emb_m3,
    )
    print(f'  ✅ bge-m3 DB loaded — {vectordb_m3._collection.count():,} chunks')
else:
    vectordb_m3 = None
    print(f'  ⚠️  bge-m3 ChromaDB not found at {CHROMA_BGE_M3}')
    print('     To build it:')
    print('       1. In config.py, set embedding_model = "BAAI/bge-m3"')
    print('       2. Delete chroma_db/ and start the app once (it will rebuild)')
    print('       3. Rename chroma_db/ → chroma_db_bge_m3/')
    print('       4. Restore embedding_model = "BAAI/bge-large-en-v1.5" in config.py')
    print('     Skipping Experiment F — only production model will be evaluated.')

In [ ]:
# ── Run 65-question stratified evaluation for both models (≈10q per language) ─
ret_prod = build_retriever(vectordb, reranker_large, BEST_THRESHOLD, BEST_POOL, BEST_TOPN)

# Stratified sample: ~10 per language (7 languages × ~10 = ~70, capped to 65)
df_eval_f = (
    df_eval.groupby('language', group_keys=False)
           .apply(lambda g: g.sample(n=min(len(g), 10), random_state=42))
           .reset_index(drop=True)
)
print(f'Exp F sample size: {len(df_eval_f)}q across {df_eval_f["language"].nunique()} languages')

print('── bge-large-en-v1.5 (production) — stratified 65q ─────────────')
df_emb_prod = run_experiment(df_eval_f, ret_prod, emb_prod,
                              temperature=BEST_TEMP, run_ragas=True)
df_emb_prod.to_csv(RESULTS_DIR / 'expF_bge_large.csv', index=False)
print(f'  Saved → {RESULTS_DIR}/expF_bge_large.csv')

df_emb_m3 = None
if vectordb_m3 is not None:
    ret_m3 = build_retriever(vectordb_m3, reranker_large, BEST_THRESHOLD, BEST_POOL, BEST_TOPN)
    print('\n── bge-m3 (alternative) — full dataset ──────────────────────────')
    df_emb_m3 = run_experiment(df_eval_f, ret_m3, emb_m3,
                                temperature=BEST_TEMP, run_ragas=True)
    df_emb_m3.to_csv(RESULTS_DIR / 'expF_bge_m3.csv', index=False)
    print(f'  Saved → {RESULTS_DIR}/expF_bge_m3.csv')

print('\n✅ Embedding experiment done')

In [ ]:
s_emb_prod = summarize(df_emb_prod, f'bge-large-en-v1.5')
rows_emb = [s_emb_prod]
if df_emb_m3 is not None:
    s_emb_m3 = summarize(df_emb_m3, 'bge-m3')
    rows_emb.append(s_emb_m3)

df_emb_cmp = pd.DataFrame(rows_emb)
print('\n📊 Embedding Model Comparison — Overall')
print(df_emb_cmp[SHOW_COLS].to_string(index=False))

if df_emb_m3 is not None:
    bar_comparison([s_emb_prod, s_emb_m3],
                   ['bge-large-en-v1.5', 'bge-m3'],
                   'Experiment F — Embedding Model Comparison (Full Dataset)',
                   'expF_embedding_comparison.png')

---
## Experiment G — Multilingual Breakdown

The system claims multilingual support through automatic query translation.
Here we verify whether quality degrades for non-English queries.

> Uses the full-dataset results from Experiment F (production model).

In [ ]:
# ── Per-language summary ──────────────────────────────────────────────────────
lang_rows = []
for lang in df_emb_prod['language'].unique():
    sub = df_emb_prod[df_emb_prod['language'] == lang]
    s   = summarize(sub, lang)
    lang_rows.append(s)

df_lang = pd.DataFrame(lang_rows).sort_values('rag_system_quality', ascending=False)
print('📊 Per-Language Breakdown (production embedding):')
print(df_lang[['label','n_total','document_coverage','hallucination_rate',
               'faithfulness','rag_system_quality','avg_response_time']].to_string(index=False))

In [ ]:
# ── Radar / grouped bar for multilingual performance ─────────────────────────
langs     = df_lang['label'].tolist()
cov_vals  = df_lang['document_coverage'].tolist()
qual_vals = df_lang['rag_system_quality'].tolist()
hall_vals = df_lang['hallucination_rate'].tolist()

x = np.arange(len(langs))
w = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w,   cov_vals,  w, label='Doc Coverage',      color=COLORS['ok'],    alpha=0.85)
ax.bar(x,       qual_vals, w, label='RAG Quality',        color=COLORS['main'],  alpha=0.85)
ax.bar(x + w,   hall_vals, w, label='Hallucination Rate', color=COLORS['warn'],  alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(langs, fontsize=10)
ax.set_ylim(0, 1.2)
ax.set_ylabel('Score')
ax.set_title('Experiment G — Multilingual Performance Breakdown', fontsize=13, fontweight='bold')
ax.axhline(y=s_emb_prod['document_coverage'],  color=COLORS['ok'],   linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=s_emb_prod['rag_system_quality'], color=COLORS['main'], linestyle='--', alpha=0.5, linewidth=1)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'expG_multilingual.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Response time per language ────────────────────────────────────────────────
rt_vals = df_lang['avg_response_time'].tolist()
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(langs, rt_vals, color=COLORS['accent'], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, rt_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{val:.1f}s', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Avg Response Time (s)')
ax.set_title('Response Time per Language (translation overhead)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'expG_response_time_lang.png', dpi=150, bbox_inches='tight')
plt.show()

### G — Interpretation

*Fill in after running:*
- English should be the best-performing language (corpus is in English).
- Compare translation overhead (response time) for non-English queries vs English.
- If Spanish/French quality drops significantly, this is a limitation to discuss in the report.
- Dashed horizontal lines show the overall average for easy comparison.

---
## 9. Final Results — Full Evaluation with Best Config

In [ ]:
# df_emb_prod already holds the full 140-question results with the best config.
# If not yet run, re-run Experiment F cell first.

df_final = df_emb_prod.copy()
s_final  = summarize(df_final, 'Best Config (Full Dataset)')

class_cols = ['label','accuracy','precision','recall','f1',
              'document_coverage','hallucination_rate','rag_system_quality','avg_response_time']
print('\n📊 FINAL RESULTS SUMMARY')
print('='*80)
print(pd.DataFrame([s_final])[class_cols].to_string(index=False))

# Save
df_final.to_csv(RESULTS_DIR / 'final_results.csv', index=False)
pd.DataFrame([s_final]).to_csv(RESULTS_DIR / 'final_summary.csv', index=False)
print(f'\n✅ Saved → {RESULTS_DIR}/final_results.csv')
print(f'✅ Saved → {RESULTS_DIR}/final_summary.csv')

In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────────
TP = s_final['TP']; FP = s_final['FP']
FN = s_final['FN']; TN = s_final['TN']
cm = np.array([[TP, FN], [FP, TN]])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', ax=ax,
            cmap=sns.light_palette(COLORS['main'], as_cmap=True),
            xticklabels=['Pred: Answered', 'Pred: Refused'],
            yticklabels=['Actual: In DB', 'Actual: Not in DB'])
ax.set_title('Confusion Matrix — Final Best Config', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Response time distribution ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_final['response_time'], bins=25, color=COLORS['main'],
        alpha=0.85, edgecolor='white')
ax.axvline(df_final['response_time'].mean(), color=COLORS['warn'],
           linestyle='--', linewidth=2,
           label=f'Mean = {df_final["response_time"].mean():.1f}s')
ax.axvline(df_final['response_time'].median(), color=COLORS['accent'],
           linestyle='--', linewidth=2,
           label=f'Median = {df_final["response_time"].median():.1f}s')
ax.set_xlabel('Response Time (s)')
ax.set_ylabel('Count')
ax.set_title('Response Time Distribution (Final Config)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_response_time_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Qualitative Analysis — Failure Cases

In [ ]:
# ── False Negatives: in-scope questions the model refused ────────────────────
fn = df_final[(df_final['label']==1) & (df_final['pred']==0)]
print(f'=== FALSE NEGATIVES (Missed) — {len(fn)} questions ===')
if len(fn):
    display(fn[['id','language','question','answer_preview']].head(10))

print()

# ── False Positives: out-of-scope questions the model answered ───────────────
fp = df_final[(df_final['label']==0) & (df_final['pred']==1)]
print(f'=== FALSE POSITIVES (Hallucinations) — {len(fp)} questions ===')
if len(fp):
    display(fp[['id','language','question','answer_preview']].head(10))

In [ ]:
# ── Docs-retrieved distribution ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
df_final['n_docs_retrieved'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color=COLORS['main'], alpha=0.85, edgecolor='white')
ax.set_xlabel('Chunks retrieved after reranking')
ax.set_ylabel('Question count')
ax.set_title('Distribution of Retrieved Chunks per Query', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_chunks_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(df_final['n_docs_retrieved'].describe())

---
## 11. Report Summary — Copy-Paste Block

Run this cell last to print the canonical numbers for your paper.

In [ ]:
BEST_THR  = BEST_THRESHOLD
BEST_POOL_V = BEST_POOL
BEST_TOPN_V = BEST_TOPN
BEST_TEMP_V = BEST_TEMP

print('='*70)
print('BABYCHATTY RAG SYSTEM — FINAL EVALUATION SUMMARY'.center(70))
print('='*70)

print('\n[1] CHOSEN CONFIGURATION')
print(f'  Embedding model    : {cfg.embedding_model}')
print(f'  Reranker model     : {cfg.re_rank_model}')
print(f'  Similarity threshold: {BEST_THR}')
print(f'  Candidate pool size: {BEST_POOL_V}  (before reranking)')
print(f'  Final top-N chunks : {BEST_TOPN_V}  (after reranking)')
print(f'  Temperature        : {BEST_TEMP_V}')
print(f'  Chunk size         : {cfg.chunk_size} chars / {cfg.chunk_overlap} overlap')
print(f'  LLM                : {cfg.model_name}')
print(f'  Judge LLM (RAGAS)  : {cfg.judge_name}')

print('\n[2] SYSTEM PERFORMANCE (full 140-question dataset)')
print(f'  Document Coverage  : {s_final["document_coverage"]:.1%}')
print(f'  Hallucination Rate : {s_final["hallucination_rate"]:.1%}')
print(f'  Avg Response Time  : {s_final["avg_response_time"]:.1f} s')
print(f'  RAG System Quality : {s_final["rag_system_quality"]:.3f}')
print(f'     Faithfulness     : {s_final["faithfulness"]:.3f}')
print(f'     Answer Relevancy : {s_final["answer_relevancy"]:.3f}')
print(f'     Context Util.    : {s_final["context_utilization"]:.3f}')
print(f'  Accuracy           : {s_final["accuracy"]:.3f}')
print(f'  F1 Score           : {s_final["f1"]:.3f}')
print(f'  Confusion Matrix   : TP={s_final["TP"]} FP={s_final["FP"]} TN={s_final["TN"]} FN={s_final["FN"]}')

print('\n[3] OUTPUT FILES')
for f in sorted(RESULTS_DIR.glob('*.csv')):
    print(f'  {f}')
for f in sorted(FIGURES_DIR.glob('*.png')):
    print(f'  {f}')

print('\n' + '='*70)

---
## Appendix: Output Files

| File | Description |
|------|-------------|
| `results/final_results.csv` | Per-question results, best config, full dataset |
| `results/final_summary.csv` | Aggregate metrics |
| `results/expE_reranker_large.csv` | Reranker-large per-question results |
| `results/expE_reranker_base.csv` | Reranker-base per-question results |
| `results/expF_bge_large.csv` | bge-large embedding per-question results |
| `results/expF_bge_m3.csv` | bge-m3 embedding per-question results (if built) |
| `figures/expA_threshold.png` | Similarity threshold ablation |
| `figures/expB_pool_size.png` | Candidate pool ablation |
| `figures/expC_topn.png` | Top-N ablation |
| `figures/expD_temperature.png` | Temperature ablation |
| `figures/expD_tradeoff_scatter.png` | Faithfulness vs Hallucination scatter |
| `figures/expE_reranker_comparison.png` | Reranker head-to-head |
| `figures/expE_reranker_time.png` | Reranker latency |
| `figures/expF_embedding_comparison.png` | Embedding model head-to-head |
| `figures/expG_multilingual.png` | Per-language metrics |
| `figures/expG_response_time_lang.png` | Per-language response time |
| `figures/final_confusion_matrix.png` | Final confusion matrix |
| `figures/final_response_time_dist.png` | Response time histogram |
| `figures/final_chunks_distribution.png` | Chunks-retrieved histogram |